In [1]:
import pandas as pd
import time
import sqlite3
from nba_api.stats.endpoints import synergyplaytypes

In [2]:
season = "2024-25"

In [3]:
from nba_api.stats.endpoints import CommonAllPlayers
import pandas as pd

# Get all players for the 2023-24 season
players_data = CommonAllPlayers(is_only_current_season=1, league_id='00')
players_df = players_data.get_data_frames()[0]

# Extract relevant player IDs and names
player_ids = players_df['PERSON_ID'].tolist()
print(f"Total players for 2023-24 season: {len(player_ids)}")

Total players for 2023-24 season: 575


In [4]:
from nba_api.stats.endpoints import LeagueGameFinder

# Set parameters for the desired season and team
gamefinder = LeagueGameFinder(season_nullable=season,season_type_nullable="Regular Season")

# Get games as a DataFrame
games = gamefinder.get_data_frames()[0]

# Optional: Filter to only include date, home and away teams, and matchup info
schedule = pd.DataFrame(games[['GAME_DATE']])

# Convert game date to a readable format if needed
schedule['GAME_DATE'] = pd.to_datetime(schedule['GAME_DATE']).dt.strftime('%m/%d/%Y')
dates = schedule.drop_duplicates()

In [5]:
dates

,GAME_DATE
0,04/13/2025
30,04/11/2025
60,04/10/2025
70,04/09/2025
90,04/08/2025
...,...
3442,10/26/2024
3462,10/25/2024
3482,10/24/2024
3490,10/23/2024


In [6]:
'''from nba_api.stats.endpoints import LeagueGameFinder

# Get all games for the 2023-24 season
gamefinder = LeagueGameFinder(season_nullable='2023-24')
games_df = gamefinder.get_data_frames()[0]

# Extract relevant game IDs
game_ids = games_df['GAME_ID'].tolist()
print(f"Total games for 2023-24 season: {len(game_ids)}")'''

'from nba_api.stats.endpoints import LeagueGameFinder\n\n# Get all games for the 2023-24 season\ngamefinder = LeagueGameFinder(season_nullable=\'2023-24\')\ngames_df = gamefinder.get_data_frames()[0]\n\n# Extract relevant game IDs\ngame_ids = games_df[\'GAME_ID\'].tolist()\nprint(f"Total games for 2023-24 season: {len(game_ids)}")'

In [7]:
conn = sqlite3.connect('nba_data.db')
existing_dates = pd.read_sql(f"SELECT GAME_DATE FROM shot_detail_data", conn)

existing_dates['GAME_DATE'] = pd.to_datetime(existing_dates['GAME_DATE'], format='%Y%m%d').dt.strftime('%m/%d/%Y')

unique_dates = dates.merge(
    existing_dates, 
    on="GAME_DATE", 
    how="left", 
    indicator=True
)

# Keep only the dates from `dates` that do not exist in `existing_dates`
unique_dates = unique_dates[unique_dates["_merge"] == "left_only"]

# Drop the '_merge' column if you don't need it
unique_dates = unique_dates.drop(columns=["_merge"])

In [8]:
last_date = unique_dates.iloc[0][0]
first_date = unique_dates.iloc[-1][0]

In [9]:
first_date

'03/14/2025'

In [10]:
players_df.loc[players_df['PERSON_ID']==1631204]

,PERSON_ID,DISPLAY_LAST_COMMA_FIRST,DISPLAY_FIRST_LAST,ROSTERSTATUS,FROM_YEAR,TO_YEAR,PLAYERCODE,PLAYER_SLUG,TEAM_ID,TEAM_CITY,TEAM_NAME,TEAM_ABBREVIATION,TEAM_SLUG,TEAM_CODE,GAMES_PLAYED_FLAG,OTHERLEAGUE_EXPERIENCE_CH
447,1631204,"Sasser, Marcus",Marcus Sasser,1,2023,2024,marcus_sasser,marcus_sasser,1610612765,Detroit,Pistons,DET,pistons,pistons,Y,00


In [11]:
players_ids_failed = [1642359, 1629640, 1642502, 1629006, 1631204]

In [12]:


from nba_api.stats.endpoints import ShotChartDetail
import time
all_shot_data = []
# Loop through each player ID to fetch shot data for the entire season
for player_id in player_ids:  # Using [:10] as a subset for testing
    attempt = 0
    success = False
    
    while attempt < 3 and not success:
        try:
            # Fetch shot chart details for the player across the entire season
            shot_data = ShotChartDetail(
                team_id=0,  # Team ID can be 0 to include all teams
                player_id=player_id,
                season_nullable=season,
                date_from_nullable=first_date,
                date_to_nullable=last_date,
                context_measure_simple = "FGA"
            )
            
            # Convert the data to a DataFrame
            shot_df = shot_data.get_data_frames()[0]
            
            # Append the DataFrame to the list
            all_shot_data.append(shot_df)
            success = True
            
            # Respect API rate limits by adding a delay
            time.sleep(5)
        
        except Exception as e:
            attempt += 1
            print(f"Attempt {attempt} failed for player {player_id}: {e}")
            time.sleep(10 * attempt)

# Combine all shot details into one DataFrame
#all_shots_df = pd.concat(all_shot_data, ignore_index=True)
all_shots_df = pd.concat(all_shot_data, ignore_index=True)



Attempt 1 failed for player 1628379: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Attempt 1 failed for player 1630256: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Attempt 1 failed for player 1628369: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)


In [13]:
all_shots_df['PLAYER_NAME'].nunique()

487

In [14]:
#failed_all_shots_df

In [15]:
#all_shots_df = pd.concat([all_shots_df,failed_all_shots_df])

In [16]:
all_shots_df['id'] = all_shots_df['PLAYER_ID'].astype(str)+all_shots_df['GAME_ID'].astype(str)+all_shots_df['SHOT_ZONE_AREA'].astype(str)+all_shots_df['GAME_EVENT_ID'].astype(str)

In [17]:
#all_failed['id'] = all_failed['PLAYER_ID'].astype(str)+all_failed['GAME_ID'].astype(str)+all_failed['SHOT_ZONE_AREA'].astype(str)+all_failed['GAME_EVENT_ID'].astype(str)
#all_shots_df =all_failed

In [18]:
#all_shots_df

In [19]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "shot_detail_data"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

In [20]:
#all_shots_df.to_sql(table_name, conn, if_exists='replace', index=False)

In [21]:
# Step 2: Create the table if it doesn't exist
if not table_exists:
    all_shots_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = all_shots_df[~all_shots_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'shot_detail_data' already exists. Checking for new records...
Inserted 43500 new records into 'shot_detail_data'.


### 